In [112]:
import torch.nn as nn
import torch
import torchmetrics
import torchvision
from torch.nn.init import kaiming_uniform_
from torch.utils.data import DataLoader

In [113]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [114]:
import torchvision
from torchvision import transforms

train_data = torchvision.datasets.CIFAR10(root="./data", train=True,  download=False, transform=transforms.ToTensor())
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=False, transform=transforms.ToTensor())

In [115]:
all_images = torch.stack([img for img, _ in train_data])
mean = all_images.mean(dim=[0, 2, 3])
std  = all_images.std(dim=[0, 2, 3])

full_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])
train_data.transform = full_transform
test_data.transform  = full_transform

In [116]:
from torch.utils.data import random_split

train_data, valid_data = random_split(train_data, [45000, 5000])

In [117]:
train_loader = DataLoader(train_data, batch_size=32, num_workers=4, pin_memory=True, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_data,   batch_size=32, num_workers=4, pin_memory=True)

In [118]:
images, labels = next(iter(train_loader))
images.shape

torch.Size([32, 3, 32, 32])

In [119]:
images, labels = next(iter(train_loader))
print(images.mean())  # should be close to 0
print(images.std())

tensor(0.1261)
tensor(1.0089)


In [120]:
def he_initialization(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
        nn.init.zeros_(module.bias)


def build_mlp(n_hidden, n_neurons, n_inputs, n_outputs, dropout_p):
    layers = [nn.Flatten(), nn.Linear(n_inputs, n_neurons), nn.BatchNorm1d(n_neurons), nn.ReLU(), nn.Dropout(dropout_p)]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(n_neurons, n_neurons), nn.BatchNorm1d(n_neurons), nn.ReLU(), nn.Dropout(dropout_p)]
    layers += [nn.Linear(n_neurons, n_outputs)]

    for module in layers:
        he_initialization(module)

    return nn.Sequential(*layers)

In [121]:
def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    model.train()
    return metric.compute().item()




def train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, n_iter_no_improvements, scheduler=None):
    metrics = []
    model.train()

    best_val_score = 0
    iter_no_improvements = 0

    for epoch in range(n_epochs):
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            epoch_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR):
                scheduler.step()

        mean_loss = epoch_loss / len(train_loader)
        metrics = evaluate(model, valid_loader, metric)
        print(f"Epoch: {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}, Val Score: {metrics:.4f}")
        if scheduler is not None and not isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR):
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(metrics)
            else:
                scheduler.step()
        if metrics > best_val_score:
            best_val_score = metrics
            iter_no_improvements = 0
        else:
            iter_no_improvements += 1
        if iter_no_improvements >= n_iter_no_improvements:
            print(f"Validation score has not improved for {n_iter_no_improvements} epochs, stopping training")
            return best_val_score
    return best_val_score


In [122]:
n_epochs = 100
model = build_mlp(n_hidden=10, n_neurons=50, n_inputs=3*32*32, n_outputs=10, dropout_p=0.3).to(device)
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=1e-2, total_steps=len(train_loader)*n_epochs)
n_iter_no_improvements = 3
train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, n_iter_no_improvements, scheduler=scheduler)

Epoch: 1/100, Loss: 2.4773, Val Score: 0.1090
Epoch: 2/100, Loss: 2.2627, Val Score: 0.1278
Epoch: 3/100, Loss: 2.1769, Val Score: 0.1316
Epoch: 4/100, Loss: 2.1225, Val Score: 0.1722
Epoch: 5/100, Loss: 2.0840, Val Score: 0.2118
Epoch: 6/100, Loss: 2.0373, Val Score: 0.2260
Epoch: 7/100, Loss: 2.0037, Val Score: 0.2366
Epoch: 8/100, Loss: 1.9812, Val Score: 0.2590
Epoch: 9/100, Loss: 1.9658, Val Score: 0.2642
Epoch: 10/100, Loss: 1.9540, Val Score: 0.2730
Epoch: 11/100, Loss: 1.9488, Val Score: 0.2880
Epoch: 12/100, Loss: 1.9444, Val Score: 0.3090
Epoch: 13/100, Loss: 1.9364, Val Score: 0.2830
Epoch: 14/100, Loss: 1.9414, Val Score: 0.3044
Epoch: 15/100, Loss: 1.9358, Val Score: 0.2952
Validation score has not improved for 3 epochs, stopping training


0.3089999854564667

In [123]:
import optuna
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR, ReduceLROnPlateau


def objective(trial):
    max_lr = trial.suggest_float("max_lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 2, 8)
    n_neurons = trial.suggest_categorical("n_neurons", [32, 64, 128, 256])
    dropout_p = trial.suggest_float("dropout_p", 0.1, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "Adam", "SGD"])
    scheduler_name = trial.suggest_categorical("scheduler", ["OneCycleLR", "CosineAnnealingLR", "ReduceLROnPlateau"])

    model = build_mlp(n_hidden, n_neurons, 3*32*32, 10, dropout_p).to(device)

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=max_lr, weight_decay=weight_decay, momentum=0.9)


    if scheduler_name == "OneCycleLR":
        scheduler = OneCycleLR(optimizer, max_lr=max_lr, total_steps=len(train_loader) * n_epochs)
    elif scheduler_name == "CosineAnnealingLR":
        scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    elif scheduler_name == "ReduceLROnPlateau":
        scheduler = ReduceLROnPlateau(optimizer, patience=3)


    criterion = nn.CrossEntropyLoss()
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

    best_val_score = train(model, optimizer, criterion, train_loader,
              valid_loader, metric, n_epochs,
              n_iter_no_improvements, scheduler=scheduler)
    return best_val_score

In [124]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

[I 2026-06-05 21:20:40,305] A new study created in memory with name: no-name-935b482e-1332-435c-84c3-0aeae3ff0dfc


Epoch: 1/100, Loss: 2.4885, Val Score: 0.1088
Epoch: 2/100, Loss: 2.2774, Val Score: 0.1126
Epoch: 3/100, Loss: 2.2025, Val Score: 0.1176
Epoch: 4/100, Loss: 2.1674, Val Score: 0.1192
Epoch: 5/100, Loss: 2.1474, Val Score: 0.1378
Epoch: 6/100, Loss: 2.1287, Val Score: 0.1420
Epoch: 7/100, Loss: 2.1138, Val Score: 0.1684
Epoch: 8/100, Loss: 2.1007, Val Score: 0.1636
Epoch: 9/100, Loss: 2.0847, Val Score: 0.1864
Epoch: 10/100, Loss: 2.0753, Val Score: 0.1946
Epoch: 11/100, Loss: 2.0595, Val Score: 0.2150
Epoch: 12/100, Loss: 2.0459, Val Score: 0.2190
Epoch: 13/100, Loss: 2.0411, Val Score: 0.2302
Epoch: 14/100, Loss: 2.0291, Val Score: 0.2264
Epoch: 15/100, Loss: 2.0228, Val Score: 0.2358
Epoch: 16/100, Loss: 2.0120, Val Score: 0.2300
Epoch: 17/100, Loss: 2.0091, Val Score: 0.2294
Epoch: 18/100, Loss: 2.0002, Val Score: 0.2456
Epoch: 19/100, Loss: 1.9970, Val Score: 0.2386
Epoch: 20/100, Loss: 1.9941, Val Score: 0.2434
Epoch: 21/100, Loss: 1.9902, Val Score: 0.2508
Epoch: 22/100, Loss: 1

[I 2026-06-05 21:22:14,203] Trial 0 finished with value: 0.26339998841285706 and parameters: {'max_lr': 0.011579782107562685, 'n_hidden': 6, 'n_neurons': 32, 'dropout_p': 0.41380489537973164, 'weight_decay': 0.00026559409562235993, 'optimizer': 'SGD', 'scheduler': 'OneCycleLR'}. Best is trial 0 with value: 0.26339998841285706.


Epoch: 26/100, Loss: 1.9752, Val Score: 0.2552
Validation score has not improved for 3 epochs, stopping training
Epoch: 1/100, Loss: 2.8404, Val Score: 0.1066
Epoch: 2/100, Loss: 2.7560, Val Score: 0.1188
Epoch: 3/100, Loss: 2.7194, Val Score: 0.1282
Epoch: 4/100, Loss: 2.6935, Val Score: 0.1380
Epoch: 5/100, Loss: 2.6720, Val Score: 0.1508
Epoch: 6/100, Loss: 2.6612, Val Score: 0.1536
Epoch: 7/100, Loss: 2.6310, Val Score: 0.1556
Epoch: 8/100, Loss: 2.6064, Val Score: 0.1488
Epoch: 9/100, Loss: 2.5917, Val Score: 0.1498


[I 2026-06-05 21:22:50,708] Trial 1 finished with value: 0.15559999644756317 and parameters: {'max_lr': 1.2569692517174708e-05, 'n_hidden': 6, 'n_neurons': 256, 'dropout_p': 0.34295488496468995, 'weight_decay': 1.3745455818533825e-05, 'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR'}. Best is trial 0 with value: 0.26339998841285706.


...

Epoch: 8/100, Loss: 1.6803, Val Score: 0.4278
Validation score has not improved for 3 epochs, stopping training
Epoch: 1/100, Loss: 1.9087, Val Score: 0.3876
Epoch: 2/100, Loss: 1.7673, Val Score: 0.4316
Epoch: 3/100, Loss: 1.7049, Val Score: 0.4424
Epoch: 4/100, Loss: 1.6742, Val Score: 0.4500
Epoch: 5/100, Loss: 1.6412, Val Score: 0.4682
Epoch: 6/100, Loss: 1.6179, Val Score: 0.4694
Epoch: 7/100, Loss: 1.5937, Val Score: 0.4780
Epoch: 8/100, Loss: 1.5724, Val Score: 0.4804
Epoch: 9/100, Loss: 1.5485, Val Score: 0.4978
Epoch: 10/100, Loss: 1.5321, Val Score: 0.4906
Epoch: 11/100, Loss: 1.5220, Val Score: 0.4754
Epoch: 12/100, Loss: 1.5117, Val Score: 0.5006
Epoch: 13/100, Loss: 1.4973, Val Score: 0.5062
Epoch: 14/100, Loss: 1.4849, Val Score: 0.5056
Epoch: 15/100, Loss: 1.4745, Val Score: 0.5096
Epoch: 16/100, Loss: 1.4707, Val Score: 0.5122
Epoch: 17/100, Loss: 1.4556, Val Score: 0.5112
Epoch: 18/100, Loss: 1.4446, Val Score: 0.5154
Epoch: 19/100, Loss: 1.4364, Val Score: 0.5166
Epoc

[I 2026-06-05 21:27:54,855] Trial 5 finished with value: 0.522599995136261 and parameters: {'max_lr': 0.01352896091356954, 'n_hidden': 3, 'n_neurons': 128, 'dropout_p': 0.2849400192835263, 'weight_decay': 6.284957386840769e-05, 'optimizer': 'AdamW', 'scheduler': 'CosineAnnealingLR'}. Best is trial 5 with value: 0.522599995136261.


Epoch: 23/100, Loss: 1.4036, Val Score: 0.5218
Validation score has not improved for 3 epochs, stopping training
Epoch: 1/100, Loss: 2.9298, Val Score: 0.1208
Epoch: 2/100, Loss: 2.9117, Val Score: 0.1210
Epoch: 3/100, Loss: 2.8969, Val Score: 0.1226
Epoch: 4/100, Loss: 2.8717, Val Score: 0.1264


[W 2026-06-05 21:28:16,317] Trial 6 failed with parameters: {'max_lr': 2.3721612008547765e-05, 'n_hidden': 6, 'n_neurons': 256, 'dropout_p': 0.4369712750959467, 'weight_decay': 0.0008793998988558985, 'optimizer': 'Adam', 'scheduler': 'OneCycleLR'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_20341/728936978.py", line 35, in objective
    best_val_score = train(model, optimizer, criterion, train_loader,
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_20341/1843366059.py", line 31, in train
    optimizer.step()
  File "/home/denys/PycharmProjects/Hands-On_ML/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py", line 140, in wrapper
    return func.__get__(opt, opt.__cl

KeyboardInterrupt: 

In [125]:
full_train_data = torch.utils.data.ConcatDataset([train_data, valid_data])
full_train_loader = DataLoader(full_train_data, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

best = study.best_params

model = build_mlp(best['n_hidden'], best['n_neurons'], 3*32*32, 10, best['dropout_p']).to(device)

if best['optimizer'] == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=best['max_lr'], weight_decay=best['weight_decay'])
elif best['optimizer'] == 'AdamW':
    optimizer = torch.optim.AdamW(model.parameters(), lr=best['max_lr'], weight_decay=best['weight_decay'])
elif best['optimizer'] == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=best['max_lr'], weight_decay=best['weight_decay'], momentum=0.9)

if best['scheduler'] == 'OneCycleLR':
    scheduler = OneCycleLR(optimizer, max_lr=best['max_lr'], total_steps=len(full_train_loader) * n_epochs)
elif best['scheduler'] == 'CosineAnnealingLR':
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
elif best['scheduler'] == 'ReduceLROnPlateau':
    scheduler = ReduceLROnPlateau(optimizer, patience=3)

criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

train(model, optimizer, criterion, full_train_loader, test_loader, metric, n_epochs, n_iter_no_improvements, scheduler=scheduler)

test_score = evaluate(model, test_loader, metric)
print(f"Test Accuracy: {test_score:.4f}")

Epoch: 1/100, Loss: 1.8915, Val Score: 0.4112
Epoch: 2/100, Loss: 1.7527, Val Score: 0.4455
Epoch: 3/100, Loss: 1.7027, Val Score: 0.4666
Epoch: 4/100, Loss: 1.6654, Val Score: 0.4730
Epoch: 5/100, Loss: 1.6319, Val Score: 0.4743
Epoch: 6/100, Loss: 1.6061, Val Score: 0.4812
Epoch: 7/100, Loss: 1.5868, Val Score: 0.4964
Epoch: 8/100, Loss: 1.5693, Val Score: 0.4814
Epoch: 9/100, Loss: 1.5468, Val Score: 0.4964
Epoch: 10/100, Loss: 1.5336, Val Score: 0.5070
Epoch: 11/100, Loss: 1.5252, Val Score: 0.5152
Epoch: 12/100, Loss: 1.5087, Val Score: 0.4998
Epoch: 13/100, Loss: 1.4900, Val Score: 0.5151
Epoch: 14/100, Loss: 1.4856, Val Score: 0.5118
Validation score has not improved for 3 epochs, stopping training
Test Accuracy: 0.5118


In [126]:
torch.save({
    "model_state_dict": model.state_dict(),
    "hyperparameters": study.best_params
}, "cifar10_mlp.pth")